In [1]:
import os
GITHUB_TOKEN=""  # @param {type: "string"}
REPO_URL = "https://github.com/NeuralFalconYT/video-text-removal.git"
# Use the token in the correct format
CLONE_URL = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")
%cd /content/
!rm -rf /content/video-text-removal
!git clone {CLONE_URL}


/content
Cloning into 'video-text-removal'...
remote: Enumerating objects: 69, done.
remote: Counting objects: 100% (69/69), done.
remote: Compressing objects: 100% (54/54), done.
Receiving objects: 100% (69/69), 30.88 MiB | 6.18 MiB/s, done.
remote: Total 69 (delta 8), reused 0 (delta 0), pack-reused 0 (from 0)
Resolving deltas: 100% (8/8), done.


In [ ]:
!pip install loguru
!pip install rapidocr_onnxruntime onnxruntime
# !pip install onnxruntime-gpu


In [5]:

%cd /content/video-text-removal

import sys
path = "/content/video-text-removal"

if path not in sys.path:
    sys.path.insert(0, path)

from mi_gan.image_inpainting import MIGAN
from mi_gan.schema import InpaintRequest
from ocr.utils import run_ocr,draw_polygons,generate_text_mask,blur_text_regions
import torch
import cv2
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
model = MIGAN(device=device)
config = InpaintRequest(
    hd_strategy="crop",
    hd_strategy_crop_margin=128,
)






def local_ocr(image,bounding_box=True,hide_method=0):
  result=image
  result_path=None
  if isinstance(image, np.ndarray):
    frame_path="./frame.png"
    img = image
  if isinstance(image, str):
    frame_path=image
    img = cv2.imread(image)
  raw_img=img.copy()
  ocr_items=run_ocr(img)
  if bounding_box:
    raw_img=draw_polygons(raw_img, ocr_items, display_text=False)
  if hide_method==1:
    method="inpaint"
  else:
    method="blur"
  if method=="blur":
    blurred_img = blur_text_regions(img,
                                  ocr_items,
                                  blur_level=3,
                                  blur_sigma=0,
                                  min_conf=0.0,
                                  expand_scale=1.0)
    result=blurred_img
    base, ext = os.path.splitext(frame_path)
    blur_image_path = f"{base}_blur{ext}"
    cv2.imwrite(blur_image_path, blurred_img)
    result_path=blur_image_path
  if method=="inpaint":
    mask =  generate_text_mask(img,
                              ocr_items,
                              invert=False,
                              min_conf=0.0,
                              expand_scale=1.0
                            )
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    output_inpaint = model(img_rgb, mask, config)
    result=output_inpaint
    base, ext = os.path.splitext(frame_path)
    mask_path = f"{base}_inpaint{ext}"
    cv2.imwrite(mask_path, output_inpaint)
    result_path=mask_path
  return raw_img,result,result_path


2025-12-17 16:22:07.532 | INFO     | mi_gan.utils:load_jit_model:81 - Loading model from: /root/.cache/torch/hub/checkpoints/migan_traced.pt


/content/video-text-removal


In [6]:
frame_path="/content/frame.png"
image,result,result_path=local_ocr(frame_path,bounding_box=True,hide_method=1)

In [ ]:
#inpaint
frame_path="/content/frame.png"
image,result,result_path=local_ocr(frame_path,bounding_box=True,hide_method=1)
from google.colab.patches import cv2_imshow
cv2_imshow(image)
cv2_imshow(result)

In [ ]:
#blur
frame_path="/content/frame.png"
image,result,result_path=local_ocr(frame_path,bounding_box=True,hide_method=0)
from google.colab.patches import cv2_imshow
cv2_imshow(image)
cv2_imshow(result)